# SpendShield Controlled Synthetic Dataset Generation

This notebook creates a bounded, reproducible research dataset after the operational MongoDB audit found no usable current transactions or verified labels.

## Objective

- Generate fictional transaction records with transparent scenario rules.
- Record provenance, label semantics, feature boundaries, and temporal splits.
- Write only to `data/synthetic/`.

## Non-goals

This notebook does not connect to MongoDB or Cassandra, modify application state, train a model, expose an API, block transactions, or claim real-world fraud performance.

## Provenance and safety

Every row is marked `synthetic=true` and `dataset_type=synthetic_research`. Identifiers use synthetic prefixes and no names, emails, phone numbers, credentials, card data, or bank data are generated. Scenario labels are generator-rule labels, not confirmed fraud labels.

In [1]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "ml" / "synthetic_dataset_generator.py").exists()
)
sys.path.insert(0, str(REPOSITORY_ROOT))

import json

from ml.synthetic_dataset_generator import (
    SyntheticDatasetConfig,
    generate_dataset,
    validate_records,
    write_dataset,
)

CONFIG = SyntheticDatasetConfig()
OUTPUT_DIR = REPOSITORY_ROOT / "data" / "synthetic"
CONFIG.to_dict()

{'total_transactions': 10000,
 'user_count': 500,
 'merchant_count': 100,
 'duration_days': 90,
 'start_date': '2024-01-01T00:00:00+00:00',
 'currency': 'INR',
 'seed': 20260912,
 'dataset_version': 'v1',
 'generator_version': '1.0.0',
 'scenario_counts': {'normal': 7600,
  'synthetic_high_amount': 800,
  'synthetic_rapid_repeat': 600,
  'synthetic_unusual_time': 600,
  'synthetic_behavior_deviation': 300,
  'synthetic_combined_pattern': 100},
 'start_timestamp': '2024-01-01T00:00:00Z',
 'end_timestamp_exclusive': '2024-03-31T00:00:00Z',
 'transactions_per_user': 20}

## Generation configuration

The default experiment uses 10,000 transactions, 500 fictional users/accounts, 100 fictional merchants, 12 project-aligned categories, INR only, and 90 UTC days. A fixed seed makes the generated rows deterministic.

In [2]:
dataset = generate_dataset(CONFIG)

summary = {
    "row_count": len(dataset.records),
    "column_count": len(dataset.records[0]),
    "scenario_distribution": dataset.manifest["scenario_distribution"],
    "split_counts": dataset.manifest["split_counts"],
    "date_range": dataset.manifest["date_range"],
    "seed": CONFIG.seed,
}
summary

{'row_count': 10000,
 'column_count': 28,
 'scenario_distribution': {'normal': 7600,
  'synthetic_behavior_deviation': 300,
  'synthetic_combined_pattern': 100,
  'synthetic_high_amount': 800,
  'synthetic_rapid_repeat': 600,
  'synthetic_unusual_time': 600},
 'split_counts': {'train': 7061, 'validation': 1481, 'test': 1458},
 'date_range': {'start': '2024-01-01T00:05:35Z',
  'end': '2024-03-30T23:55:20Z',
  'configured_start': '2024-01-01T00:00:00Z',
  'configured_end_exclusive': '2024-03-31T00:00:00Z',
  'timezone': 'UTC',
  'timestamp_precision': 'second'},
 'seed': 20260912}

In [3]:
validation = validate_records(
    dataset.records,
    config=CONFIG,
    splits=dataset.splits,
    manifest=dataset.manifest,
)
assert validation["valid"], validation
written_manifest = write_dataset(dataset, OUTPUT_DIR, CONFIG)
{
    "output_directory": str(OUTPUT_DIR),
    "validation": validation,
    "manifest_created_at_utc": written_manifest["created_at_utc"],
}

{'output_directory': 'D:\\E_Drive\\Project\\data\\synthetic',
 'validation': {'valid': True,
  'errors': [],
  'row_count': 10000,
  'column_count': 28,
  'user_count': 500,
  'account_count': 500,
  'merchant_count': 100,
  'category_count': 12,
  'scenario_distribution': {'normal': 7600,
   'synthetic_behavior_deviation': 300,
   'synthetic_combined_pattern': 100,
   'synthetic_high_amount': 800,
   'synthetic_rapid_repeat': 600,
   'synthetic_unusual_time': 600},
  'split_counts': {'train': 7061, 'validation': 1481, 'test': 1458}},
 'manifest_created_at_utc': '2026-09-12T06:12:12Z'}

## Generation conclusion

The output is ready for controlled data-quality, feature-engineering, and temporal-baseline experiments only after the validation notebook passes. The labels remain synthetic scenario labels; this phase does not establish a fraud target or production model.